# Linguistic Interpretability Prototype

This notebook implements a **small prototype** inspired by the survey *Linguistic Interpretability of Transformer-based Language Models: a systematic review* (López‑Otal et al., 2025).

We demonstrate two common approaches:

1. **Layer-wise probing**: train a lightweight probe to predict **UPOS (POS tags)** from hidden states extracted at each layer.
2. **Attention inspection**: visualize attention heatmaps for selected layers/heads.

> هدف: نمایش کوچک و عملی از ایده‌های مقاله (نه بازتولید جدول خاص، چون مقاله Survey است).

## 0) Setup

Run once (if needed):

```bash
pip install -r requirements.txt
```

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# (Optional) for extra determinism:
# import torch
# torch.manual_seed(SEED)


## 1) Choose language + dataset

We use UD via 🤗 `datasets`.

Common options:
- English: `en_ewt`
- Persian: `fa_seraji`

You can switch `UD_CONFIG` below.

In [ ]:
UD_CONFIG = "en_ewt"      # change to "fa_seraji" for Persian
MAX_SENTENCES = 200      # keep small for CPU
MODEL_NAME = "bert-base-multilingual-cased"
DEVICE = "cpu"

LAYERS = list(range(0, 13))  # 0..12 for BERT-base (0=embeddings)
DROP_UPOS = ("PUNCT",)       # optional: drop punctuation tokens


In [ ]:
from src.ud import load_ud, flatten_tokens_and_labels

train_sents = load_ud(UD_CONFIG, split="train", max_sentences=MAX_SENTENCES, seed=SEED)
try:
    eval_sents = load_ud(UD_CONFIG, split="validation", max_sentences=MAX_SENTENCES, seed=SEED)
    if len(eval_sents) == 0:
        raise ValueError("Empty validation split")
except Exception:
    eval_sents = load_ud(UD_CONFIG, split="test", max_sentences=MAX_SENTENCES, seed=SEED)

train_tokens, train_upos = flatten_tokens_and_labels(train_sents)
eval_tokens, eval_upos = flatten_tokens_and_labels(eval_sents)

len(train_tokens), len(eval_tokens)

## 2) Load model

We extract **hidden states** layer-by-layer from a Transformer encoder (mBERT).

In [ ]:
from src.extract import load_model_and_tokenizer

tokenizer, model = load_model_and_tokenizer(MODEL_NAME, device=DEVICE)
tokenizer.__class__, model.__class__

## 3) Build token-level matrices (word-level pooled)

We:
- tokenize with `is_split_into_words=True`
- pool subword pieces to **word-level vectors** by mean
- build `X_layer` for each layer and `y` (UPOS) labels

In [ ]:
from src.probe import build_token_level_matrices

Xtr_by_layer, ytr_str = build_token_level_matrices(
    train_tokens, train_upos, tokenizer, model, layers=LAYERS, device=DEVICE, drop_upos=DROP_UPOS
)

Xev_by_layer, yev_str = build_token_level_matrices(
    eval_tokens, eval_upos, tokenizer, model, layers=LAYERS, device=DEVICE, drop_upos=DROP_UPOS
)

{layer: X.shape for layer, X in list(Xtr_by_layer.items())[:3]}, ytr_str.shape, yev_str.shape

## 4) Train + evaluate linear probes layer-wise

We train a simple **linear probe** (`SGDClassifier`) per layer, then evaluate on the held-out split.

Expected qualitative pattern (from many studies): syntactic signals often peak in **middle layers** for BERT-like models (though it varies by language/model).

In [ ]:
from src.probe import train_and_eval_layerwise_probes

results, probes, label_encoder = train_and_eval_layerwise_probes(
    Xtr_by_layer, ytr_str, Xev_by_layer, yev_str, seed=SEED
)

df = pd.DataFrame([r.__dict__ for r in results]).sort_values("layer")
df.head()

In [ ]:
plt.figure()
plt.plot(df["layer"], df["accuracy"], marker="o")
plt.axhline(df["majority_baseline"].iloc[0], linestyle="--")
plt.xlabel("Layer")
plt.ylabel("Token-level UPOS accuracy")
plt.title(f"Layer-wise POS probing | UD={UD_CONFIG} | Model={MODEL_NAME}")
plt.tight_layout()
plt.show()

best = df.loc[df["accuracy"].idxmax()]
best

In [ ]:
# Save outputs
out_dir = "demo/outputs"
os.makedirs(out_dir, exist_ok=True)

csv_path = os.path.join(out_dir, f"{UD_CONFIG}_probe_results.csv")
df.to_csv(csv_path, index=False)

import joblib
joblib_path = os.path.join(out_dir, f"{UD_CONFIG}_probes.joblib")
joblib.dump({"probes": probes, "label_encoder": label_encoder, "model": MODEL_NAME, "layers": LAYERS}, joblib_path)

csv_path, joblib_path

## 5) Attention visualization (qualitative)

Attention maps are **not guaranteed explanations**, but they are a common *inspection tool*.

Below we visualize a single head from a chosen layer for an example sentence.

In [ ]:
from src.extract import extract_hidden_states_word_level
from src.viz import plot_attention_heatmap

example_tokens = ["The", "boy", "did", "not", "play", "the", "piano", "."]
# For Persian, try: ["علی", "کتاب", "را", "به", "مریم", "داد", "."]

layer_to_show = 6   # 1..12 for attention layers (layer 0 has no attention)
head_to_show = 0

ext = extract_hidden_states_word_level(
    example_tokens,
    tokenizer,
    model,
    layers=[0],  # we only need attentions now
    device=DEVICE,
    return_attentions=True
)

attn = ext.attn_by_layer[layer_to_show][head_to_show]  # (seq_len, seq_len)
tokens_model = ext.model_tokens

plot_attention_heatmap(attn, tokens_model, title=f"Attention heatmap | layer={layer_to_show} head={head_to_show}")

## 6) Quick takeaways

- The **accuracy-by-layer curve** gives a compact, quantitative view of where POS-relevant information is most accessible to a *linear* probe.
- The best layer can differ by language/model, but many studies (esp. BERT-family) often see strong syntactic signals in **middle layers**.
- Attention maps can be inspected, but interpreting them causally requires extra care.

Next steps (optional):
- repeat for multiple UD languages and compare curves
- try a different PLM (e.g., RoBERTa, XLM-R)
- add a simple baseline (static embeddings) for comparison
